<a href="https://colab.research.google.com/github/NesrineTahmi/DeepChem-Symbolic-Regression-Prototyping/blob/main/DeepChem_symbolicAI_gsoc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class SymbolicLayer(nn.Module):
  def __init__(self, input_dim) -> None:
    super().__init__()
    self.operators = [torch.sin, torch.square, torch.exp, torch.sigmoid]
    n_ops = len(self.operators)

    # weights represent the probability of picking an operator
    self.weights = nn.Parameter(torch.randn(input_dim, n_ops))
    self.temp = 1.0 # Temperature for selection

  def forward(self, x):
    # We use Softmax to 'softly' pick which operator is best
    # As temp decreases, it picks only ONE operator (Hard selection)
    gate = F.softmax(self.weights / self.temp, dim=-1)

    outputs = []
    for i, op in enumerate(self.operators):
        outputs.append(op(x) * gate[:, i])

    return sum(outputs)

In [3]:
model = SymbolicLayer(input_dim=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
x_train = torch.linspace(-3, 3, 100).view(-1, 1)
y_target = torch.sin(x_train)

for epoch in range(200):
    optimizer.zero_grad()
    y_pred = model(x_train)
    loss = F.mse_loss(y_pred, y_target)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 17.98909568786621
Epoch 50, Loss: 0.043505340814590454
Epoch 100, Loss: 0.02400033175945282
Epoch 150, Loss: 0.016667615622282028
